# 🎨 AI Horde 실시간 이미지 채굴기 & 라이브 갤러리 뷰어 (Google Colab T4 GPU)

이 노트북을 실행하면 구글 코랩의 고성능 GPU로 **AI Horde 이미지 워커**가 가동됩니다.

### ✨ 특별 기능
1. **실시간 이미지 감상**: 전 세계 유저들이 요청해서 내 코랩이 그려준 일러스트를 **화면에 즉시 큼직하게 띄워줍니다.**
2. **프롬프트 확인**: 다른 사람들이 어떤 기상천외한 프롬프트로 그림을 주문했는지 한눈에 볼 수 있습니다.
3. **자동 파일 저장**: 생성된 모든 이미지가 `horde_images/` 폴더에 고화질로 자동 저장됩니다.
4. **폭풍 Kudos 적립**: 텍스트보다 훨씬 많은 Kudos(장당 수십~수백 포인트)가 실시간으로 본인 계정에 쌓입니다!

---
### 📌 사용 방법
1. 상단 메뉴에서 **[런타임] → [런타임 유형 변경] → 하드웨어 가속기: [T4 GPU]** 확인
2. 아래 **[1단계]**부터 순서대로 셀 재생(▶) 클릭!

In [ ]:
# [1단계] 고속 이미지 생성 필수 라이브러리 설치 (약 30초 소요)
!pip install -q diffusers transformers accelerate safetensors requests pillow
print("✅ 필수 라이브러리 설치 완료!")

In [ ]:
# [2단계] AI Horde 인기 1위 고화질 일러스트 모델(DreamShaper) 로드 (약 30초 소요)
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import os

print("📥 DreamShaper AI 모델 로드 중...")
pipe = StableDiffusionPipeline.from_pretrained(
    "Lykon/DreamShaper",
    torch_dtype=torch.float16,
    safety_checker=None
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.to("cuda")

os.makedirs("horde_images", exist_ok=True)
print("🔥 GPU 가속 모델 준비 완료! 준비 끝!")

In [ ]:
# [3단계] 실시간 이미지 채굴 & 라이브 갤러리 시작!
# ⚠️ 아래 API_KEY에 발급받으신 본인의 AI Horde 키를 넣으세요.
API_KEY = "L4qKfKC37bLUDD5hgOSeSg"
WORKER_NAME = "YeolPhaCham_Dreamer_01"  # (주의: 'colab' 단어 포함 금지)
MODEL_NAME = "Dreamshaper"

import requests
import io
import base64
import time
from PIL import Image
from IPython.display import display, HTML, clear_output

print(f"🚀 [AI Horde 이미지 채굴 시작] 워커명: {WORKER_NAME}")
print("💡 전 세계에서 들어오는 그림 요청을 기다리는 중입니다...")

total_images = 0
total_kudos = 0

while True:
    try:
        # 1. 새 작업 가져오기 (POP)
        pop_res = requests.post(
            "https://aihorde.net/api/v2/generate/pop",
            headers={"Content-Type": "application/json", "apikey": API_KEY},
            json={
                "name": WORKER_NAME,
                "models": [MODEL_NAME],
                "priority_usernames": []
            },
            timeout=30
        )
        
        if pop_res.status_code != 200:
            time.sleep(5)
            continue
            
        data = pop_res.json()
        job_id = data.get("id")
        payload = data.get("payload")
        
        if not job_id or not payload:
            time.sleep(3)
            continue
            
        # 2. 프롬프트 및 설정 파싱
        prompt = payload.get("prompt", "")
        neg_prompt = ""
        if "###" in prompt:
            parts = prompt.split("###")
            prompt = parts[0].strip()
            neg_prompt = parts[1].strip()
            
        width = min(int(payload.get("width", 512)), 768)
        height = min(int(payload.get("height", 512)), 768)
        steps = min(int(payload.get("ddim_steps", 20)), 35)
        cfg = float(payload.get("cfg_scale", 7.0))
        seed = int(payload.get("seed", 12345)) % (2**31)
        
        generator = torch.Generator("cuda").manual_seed(seed)
        
        # 3. 초고속 이미지 생성
        start_t = time.time()
        image = pipe(
            prompt=prompt,
            negative_prompt=neg_prompt,
            width=width,
            height=height,
            num_inference_steps=steps,
            guidance_scale=cfg,
            generator=generator
        ).images[0]
        gen_time = round(time.time() - start_t, 2)
        
        # 4. 이미지 WebP 압축 및 Base64 인코딩
        buf = io.BytesIO()
        image.save(buf, format="WEBP", quality=85)
        b64_str = base64.b64encode(buf.getvalue()).decode("utf-8")
        
        # 5. 로컬 horde_images 폴더에 파일 저장
        save_filename = f"horde_images/{int(time.time())}_{job_id[:8]}.webp"
        image.save(save_filename, format="WEBP", quality=90)
        
        # 6. R2 다이렉트 업로드 (URL 지원 시)
        r2_url = data.get("r2_upload")
        if r2_url:
            try:
                requests.put(r2_url, data=buf.getvalue(), headers={"Content-Type": "image/webp"}, timeout=30)
            except Exception:
                pass
                
        # 7. AI Horde 서버에 완료 결과 제출 (Kudos 획득)
        sub_res = requests.post(
            "https://aihorde.net/api/v2/generate/submit",
            headers={"Content-Type": "application/json", "apikey": API_KEY},
            json={
                "id": job_id,
                "generation": b64_str,
                "seed": seed
            },
            timeout=30
        )
        
        reward = 0
        if sub_res.status_code == 200:
            reward = sub_res.json().get("reward", 0)
            total_kudos += reward
            
        total_images += 1
        
        # 8. 🌟 화면에 생성된 이미지와 프롬프트 즉시 출력!
        print("=" * 70)
        print(f"🖼️ [생성 #{total_images}] 생성 시간: {gen_time}초 | 획득 Kudos: +{reward} (누적: {total_kudos})")
        print(f"📝 요청 프롬프트: {prompt[:120]}...")
        if neg_prompt:
            print(f"🚫 네거티브: {neg_prompt[:60]}...")
        print(f"💾 저장 파일: {save_filename} ({width}x{height})")
        display(image)
        print("=" * 70)
        
    except KeyboardInterrupt:
        print("\n⏹️ 채굴을 중단했습니다.")
        break
    except Exception as e:
        print(f"⚠️ 일시적 오류: {e}")
        time.sleep(3)